In [1]:
import sys
import os
import cv2
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../src"))

from praproceses import praproses, simpan_param
from utils_cv import baca, kontras_bercak

param = {"gamma": 1.0, "clahe_clip": 2.0, "penapis": "median", "kernel": 3}
simpan_param(param, "../hasil/param_praproses.json")

KANAL = 2  # ganti sesuai kanal terbaik Anda di Bagian 2
# (indeks BGR: 0 = B, 1 = G, 2 = R)
label = pd.read_csv("../data/sintetis/label.csv")
baris = []

for _, d in label[label.kelas != "sehat"].iterrows():
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{d.nama_file}", 0)
    out = praproses(img, param)
    baris.append([
        d.kondisi_cahaya,
        kontras_bercak(img[:, :, KANAL], md, mb),
        kontras_bercak(out[:, :, KANAL], md, mb)
    ])

evaluasi = pd.DataFrame(baris, columns=["kondisi", "KB_sebelum", "KB_sesudah"])
print(evaluasi.groupby("kondisi").mean().round(2))

                KB_sebelum  KB_sesudah
kondisi                               
bayangan              5.06        8.83
gelap                 9.88       16.03
kontras_rendah        3.47        4.74
normal                8.96       10.31
terang                6.56        5.98
